# Alignment Bias Eval — GPU pipeline

End-to-end run of the three GPU stages behind the paper:

1. **Logit benchmarks** — CrowS-Pairs, StereoSet, BBQ, IAT across all 27 base/instruct pairs
2. **Linear probing** — gender probes on the 8-pair subset (16 checkpoints)
3. **INLP + LEACE intervention** — projection-based ablation + sanity-gated re-scoring

Total wall time: **~200-250 GPU-hours** on an NVIDIA RTX PRO 6000 Blackwell (G4, 96 GB).
All three scripts are resume-safe: re-running skips any (model × benchmark) cell whose output JSON already exists, so it's safe to interrupt and restart.

## 0 · Setup

Detects Colab vs local, installs the package with GPU extras into the running kernel, and adds `src/` to `sys.path` so direct imports (`from biaseval import ...`) work without restarting the kernel.

In [ ]:
import importlib, os, subprocess, sys

IN_COLAB = 'google.colab' in sys.modules
print(f'Environment: {"Colab" if IN_COLAB else "local"}')
print(f'Python:      {sys.version.split()[0]}')

# Use sys.executable (not bare `pip`) so the install lands in the running
# kernel — avoids the classic Colab kernel/system-python mismatch.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[gpu,tracking]'])

sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
importlib.invalidate_caches()

import biaseval
print(f'biaseval:    {biaseval.__version__}')

In [ ]:
import torch
assert torch.cuda.is_available(), 'No CUDA device — switch the Colab runtime to GPU before continuing.'
props = torch.cuda.get_device_properties(0)
print(f'GPU: {props.name} ({props.total_memory / 1e9:.1f} GB)')

## 1 · Hugging Face authentication

Several models in the registry are gated (Llama 2, Llama 3.1, Gemma 2/3). Accept the license on each model's HF page first, then run the cell below to log in. Skip this cell if `HF_TOKEN` is already exported in your environment.

In [ ]:
from huggingface_hub import login
login()  # paste an access token with `read` scope when prompted

## 2 · Stage 1 — logit benchmarks (~150 GPU-hours)

Scores CrowS-Pairs, StereoSet, BBQ, and IAT across all 27 base/instruct pairs in both prompt modes (raw and chat-templated). Outputs land in `results/logit_scores/<benchmark>/<model_id>__<prompt_mode>.json`.

Pass `--family` and `--variant` to slice the run — useful for partitioning across machines.

In [ ]:
!python scripts/run_benchmarks.py

## 3 · Stage 2 — linear probing (~30 GPU-hours)

Extracts residual-stream activations on the WinoBias-40 occupation probes and fits per-layer logistic probes. Restricted to the 8-pair probing subset (16 checkpoints).

In [ ]:
!python scripts/run_probing.py

## 4 · Stage 3 — INLP + LEACE intervention (~50 GPU-hours)

Fits INLP and LEACE projection matrices on cached activations, then re-scores CrowS-Pairs and StereoSet under the forward-hook ablation. Sanity gates (post-probe accuracy ≤ 0.55, perplexity ratio ≤ 1.5) are enforced inside the script and flagged in the output JSONs.

In [ ]:
!python scripts/run_intervention.py

## 5 · Aggregate raw JSONs into parquets

Walks `results/` and writes the three long-format parquets that the analysis scripts consume: `aggregated/{logit,probe,intervention}.parquet`.

In [ ]:
!python scripts/aggregate.py

## 6 · Verification gate

Recomputes every number cited in the paper from the aggregated parquets and asserts each against the reported value. Green pytest output means the pipeline reproduced end-to-end.

In [ ]:
!pytest tests/test_paper_numbers.py -v